[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C02_Post_Training_Course/04_dpo_family/04_dpo_family.ipynb)

# 04 · DPO 家族：推导与实跑

配套讲解：`04_讲解.html` ｜ 课程：后训练与对齐 ｜ <span style="color:#888">CPU 即可，全程 < 2 分钟（可选 trl 重型 cell 除外）</span>

本 notebook 在一个**完全可控的玩具设定**里复现 DPO 家族的核心现象（沿用模块 03 的设定）：

- **玩具世界**：词表只有 20 个 token，模型做**单步生成**——policy 就是一个 20 维 categorical 分布（一组 logits）。一切量都能精确计算，没有任何近似。
- **实验 1**：构造 Bradley–Terry 偏好对数据，纯 torch 实现 `dpo_loss / ipo_loss / simpo_loss`，从同一 SFT 起点分别训练，对比 margin 曲线与最终 policy 分布。
- **实验 2**：复现讲解第 3 节的 **chosen 概率下降**现象——$\log\pi(y_w)$ 与 $\log\pi(y_l)$ 双双下跌、但差距拉大。
- **实验 3**：$\beta$ 扫描（0.01 / 0.1 / 0.5）看 KL$(\pi\|\pi_{\text{ref}})$ 的变化 + 闭式最优解 $\pi^*\propto\pi_{\text{ref}}e^{r/\beta}$ 的解析 KL 曲线。
- **可选**：trl `DPOTrainer` + Qwen2.5-0.5B-Instruct 真实模型最小实跑。
- ✏️ 3 道练习 + assert 自动判分 + 📖 参考答案。

回忆讲解第 2 节的核心公式（$h$ 为对数比差）：

$$\mathcal{L}_{\mathrm{DPO}} = -\log\sigma(\beta h),\qquad h = \log\tfrac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \log\tfrac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}$$

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)

# ---------- 玩具世界：20 词表、单步生成（接模块 03 设定） ----------
V = 20
r_true = torch.zeros(V)          # 潜在真实奖励（标注者心中的偏好强度）
r_true[0:2]  =  4.0              # 2 个「最优」token
r_true[2:8]  =  1.5              # 6 个「不错」token
r_true[8:20] = -2.0              # 12 个「差」token

# SFT/参考 policy：已经向好 token 倾斜了一些（模拟 SFT 之后的起点）
ref_logits   = 0.5 * r_true + 0.1 * torch.randn(V)
ref_logp_all = F.log_softmax(ref_logits, dim=0)   # 冻结，永不更新

# ---------- 构造偏好对：从 ref 采样两个回答，按 BT 模型标注胜负 ----------
N = 400
g = torch.Generator().manual_seed(42)
ref_probs = ref_logp_all.exp()
y1 = torch.multinomial(ref_probs, N, replacement=True, generator=g)
y2 = torch.multinomial(ref_probs, N, replacement=True, generator=g)
keep = y1 != y2                                    # 去掉自己 vs 自己
y1, y2 = y1[keep], y2[keep]

# Bradley–Terry：P(y1 胜) = sigmoid(r(y1) - r(y2))，按此概率抽签 → 标注自带噪声
p_y1_wins = torch.sigmoid(r_true[y1] - r_true[y2])
wins      = torch.bernoulli(p_y1_wins, generator=g).bool()
chosen    = torch.where(wins, y1, y2)
rejected  = torch.where(wins, y2, y1)

print(f"偏好对数量: {len(chosen)}")
print("token 被选为 chosen 的次数 :", torch.bincount(chosen,   minlength=V).tolist())
print("token 被选为 rejected 次数:", torch.bincount(rejected, minlength=V).tolist())
print("注意：好 token（2~7 号）既出现在 chosen 也出现在 rejected ——")
print("     「比差的好、比最优的差」，这正是后面 chosen 概率下降的伏笔。")

## 实验 1 ：DPO / IPO / SimPO 同台对比

三个损失（单步生成里 $|y|=1$，SimPO 的长度归一化退化为原始 log-prob）：

| 方法 | 损失 | 备注 |
|---|---|---|
| DPO  | $-\log\sigma(\beta h)$ | logistic：margin 越大越好，永不满足 |
| IPO  | $\big(h - \tfrac{1}{2\beta}\big)^2$ | 平方：把 $h$ **回归**到固定目标 $\tfrac{1}{2\beta}$ |
| SimPO | $-\log\sigma\big(\beta\,[\log\pi_\theta(y_w)-\log\pi_\theta(y_l)] - \gamma\big)$ | 无 ref 模型 + 目标 margin $\gamma$ |

三者从**同一个 SFT 起点**（`ref_logits` 的拷贝）出发训练，公平对比。预期：

- DPO 的 margin **无界增长**（logistic 永远嫌不够）；
- IPO 的 margin **饱和**（回归目标拉住了它）；
- SimPO 没有 KL 锚点，但 sigmoid + $\gamma$ 也会让它在玩具任务上饱和。

In [ ]:
# ---------- 纯 torch 实现三个损失 ----------
def dpo_loss(logp_c, logp_r, ref_logp_c, ref_logp_r, beta):
    # margin = beta * h，logsigmoid 保证数值稳定（练习 1 会让你自己写一遍）
    margin = beta * ((logp_c - ref_logp_c) - (logp_r - ref_logp_r))
    return -F.logsigmoid(margin).mean()

def ipo_loss(logp_c, logp_r, ref_logp_c, ref_logp_r, beta):
    h = (logp_c - ref_logp_c) - (logp_r - ref_logp_r)
    return ((h - 1.0 / (2.0 * beta)) ** 2).mean()

def simpo_loss(logp_c, logp_r, beta, gamma):
    # 无 ref；|y|=1 时长度归一化即原始 logp
    return -F.logsigmoid(beta * (logp_c - logp_r) - gamma).mean()

def train(method, beta, steps=400, lr=0.05, gamma=1.0):
    # 从同一 SFT 起点出发；policy = 一组可训练 logits
    logits = ref_logits.clone().requires_grad_(True)
    opt = torch.optim.Adam([logits], lr=lr)
    hist = {"raw_margin": [], "logp_c": [], "logp_r": [], "kl": []}
    for _ in range(steps):
        logp_all = F.log_softmax(logits, dim=0)
        lc, lr_ = logp_all[chosen], logp_all[rejected]
        rc, rr  = ref_logp_all[chosen], ref_logp_all[rejected]
        if method == "dpo":
            loss = dpo_loss(lc, lr_, rc, rr, beta)
        elif method == "ipo":
            loss = ipo_loss(lc, lr_, rc, rr, beta)
        else:
            loss = simpo_loss(lc, lr_, beta, gamma)
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            logp_all = F.log_softmax(logits, dim=0)
            hist["raw_margin"].append((logp_all[chosen] - logp_all[rejected]).mean().item())
            hist["logp_c"].append(logp_all[chosen].mean().item())
            hist["logp_r"].append(logp_all[rejected].mean().item())
            hist["kl"].append((logp_all.exp() * (logp_all - ref_logp_all)).sum().item())
    return logits.detach(), hist

pol_dpo,   h_dpo   = train("dpo",   beta=0.1)
pol_ipo,   h_ipo   = train("ipo",   beta=0.1)
pol_simpo, h_simpo = train("simpo", beta=2.0, gamma=1.0)   # SimPO 惯用更大的 beta

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(h_dpo["raw_margin"],   label="DPO (beta=0.1)")
ax.plot(h_ipo["raw_margin"],   label="IPO (beta=0.1)")
ax.plot(h_simpo["raw_margin"], label="SimPO (beta=2, gamma=1)")
ax.set_xlabel("step"); ax.set_ylabel("mean log pi(y_w) - log pi(y_l)")
ax.set_title("Margin curves: DPO unbounded, IPO/SimPO saturate"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
x = torch.arange(V); w = 0.2
for i, (name, pol) in enumerate([("ref/SFT", ref_logits), ("DPO", pol_dpo),
                                 ("IPO", pol_ipo), ("SimPO", pol_simpo)]):
    ax.bar(x + (i - 1.5) * w, F.softmax(pol, dim=0), width=w, label=name)
ax.set_xlabel("token id (0-1 best, 2-7 good, 8-19 bad)"); ax.set_ylabel("pi(token)")
ax.set_title("Final policy distributions"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

for name, pol in [("ref", ref_logits), ("DPO", pol_dpo), ("IPO", pol_ipo), ("SimPO", pol_simpo)]:
    p = F.softmax(pol, dim=0)
    print(f"{name:6s} P(最优两个 token) = {p[:2].sum().item():.3f}")

## 实验 2 + 3 ：训练动态——chosen 概率下降 & β 扫描

**实验 2（chosen 概率下降）**：DPO loss 只看 margin。好 token（2~7 号）在数据里**既当过 chosen 又当过 rejected**，
优化器发现把概率质量全部搬到最优 token（0~1 号）上能同时改善所有 pair 的 margin——
于是 $\log\pi(y_w)$ 和 $\log\pi(y_l)$ **一起下跌**、差距却越拉越大。这正是讲解 3.1 节的 likelihood displacement。

**实验 3（β 扫描）**：$\beta$ 是 KL 约束强度的旋钮，但要分两层看：

1. **解析层**（右下图）：闭式最优解 $\pi^*\propto\pi_{\text{ref}}\,e^{r/\beta}$ 的 KL$(\pi^*\|\pi_{\text{ref}})$ 随 $\beta$ **严格单调递减**——$\beta$ 越小，最优 policy 离参考越远。
2. **优化层**（中图，固定 150 步预算）：$\beta$ 小 → KL 漂得更远；但注意梯度尺度 $\propto\beta$，
   $\beta=0.01$ 训练也更**慢**——若训练步数拉得很长，慢与远两个效应会纠缠，曲线次序可能交叉。看训练曲线时要把「目标」和「优化速度」分开想。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) chosen 概率下降：双曲线同降、差距拉大（用实验 1 的 DPO 训练记录）
ax = axes[0]
ax.plot(h_dpo["logp_c"], label="log pi(chosen)",  color="tab:green")
ax.plot(h_dpo["logp_r"], label="log pi(rejected)", color="tab:red")
ax.set_xlabel("step"); ax.set_ylabel("mean log-prob")
ax.set_title("DPO: BOTH log-probs drop, gap widens"); ax.legend(); ax.grid(alpha=0.3)
print(f"log pi(chosen):   {h_dpo['logp_c'][0]:.2f} -> {h_dpo['logp_c'][-1]:.2f}   (下降！)")
print(f"log pi(rejected): {h_dpo['logp_r'][0]:.2f} -> {h_dpo['logp_r'][-1]:.2f}   (降得更快)")
print(f"margin:           {h_dpo['raw_margin'][0]:.2f} -> {h_dpo['raw_margin'][-1]:.2f}   (持续拉大)")

# (b) beta 扫描：固定 150 步预算下 KL(pi || pi_ref) 的轨迹
ax = axes[1]
for b in [0.01, 0.1, 0.5]:
    _, h = train("dpo", beta=b, steps=150, lr=0.02)
    ax.plot(h["kl"], label=f"beta={b}")
ax.set_xlabel("step"); ax.set_ylabel("KL(pi || pi_ref)")
ax.set_title("DPO training: smaller beta -> larger drift"); ax.legend(); ax.grid(alpha=0.3)

# (c) 解析最优解 pi* prop pi_ref * exp(r/beta) 的 KL：严格单调
ax = axes[2]
betas = torch.logspace(-2, 1, 40)
kls = []
for b in betas:
    star = F.log_softmax(ref_logp_all + r_true / b, dim=0)
    kls.append((star.exp() * (star - ref_logp_all)).sum().item())
ax.plot(betas, kls)
ax.set_xscale("log"); ax.set_xlabel("beta"); ax.set_ylabel("KL(pi* || pi_ref)")
ax.set_title("Analytic optimum: KL monotone in beta"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## （可选·重型）trl `DPOTrainer` + 真实模型最小实跑

<span style="color:#b80">⚠️ 资源预估：下载 Qwen2.5-0.5B-Instruct 约 **1GB**；纯 CPU 训 4 步约 **3–10 分钟**；需要 `pip install trl peft datasets`。</span>
跳过此 cell **不影响**本 notebook 其余内容。

把 `RUN_HEAVY` 改为 `True` 即可实跑：4 条内嵌偏好对 + LoRA + 4 步 DPO，
然后打印 trl 的核心训练指标——`rewards/chosen`、`rewards/rejected`、`rewards/margins`、`rewards/accuracies`。
对照讲解第 5 节："margins 上升、两个 rewards 同时变负" 是正常形态。

In [ ]:
RUN_HEAVY = False   # 改成 True 实跑（下载 ~1GB，CPU 较慢）

if not RUN_HEAVY:
    print("已跳过重型 cell（RUN_HEAVY=False）。")
else:
    try:
        from datasets import Dataset
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import LoraConfig
        from trl import DPOConfig, DPOTrainer

        model_id = "Qwen/Qwen2.5-0.5B-Instruct"
        tok = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(model_id)

        # 4 条内嵌偏好对：chosen 简洁直答，rejected 跑题/含糊
        ds = Dataset.from_dict({
            "prompt": [
                "用一句话解释什么是过拟合。",
                "Python 里如何反转一个列表？",
                "太阳系有几颗行星？",
                "用一句话解释什么是学习率。",
            ],
            "chosen": [
                "过拟合是模型把训练集的噪声也背了下来，导致在新数据上表现变差。",
                "用切片 lst[::-1]，或原地调用 lst.reverse()。",
                "8 颗：水星、金星、地球、火星、木星、土星、天王星、海王星。",
                "学习率是每次参数更新的步长，控制梯度下降一步走多远。",
            ],
            "rejected": [
                "过拟合嘛，这个问题很复杂，涉及很多因素，不好一概而论。",
                "列表是 Python 中一种非常重要的数据结构，用途广泛。",
                "宇宙浩瀚无垠，行星数量难以计数。",
                "学习率非常重要，调好了模型就好，调不好就不好。",
            ],
        })

        peft_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
                              task_type="CAUSAL_LM")
        cfg = DPOConfig(output_dir="dpo_qwen_toy", per_device_train_batch_size=2,
                        max_steps=4, learning_rate=5e-6, beta=0.1,
                        max_length=256, max_prompt_length=128,
                        logging_steps=1, report_to="none")
        # 传入 peft_config 且不传 ref_model 时，trl 自动用「关掉 adapter 的同一模型」当参考
        trainer = DPOTrainer(model=model, args=cfg, train_dataset=ds,
                             processing_class=tok, peft_config=peft_cfg)
        trainer.train()

        print("\n=== trl 训练指标（对照讲解第 5 节） ===")
        for entry in trainer.state.log_history:
            keys = ["rewards/chosen", "rewards/rejected", "rewards/margins", "rewards/accuracies"]
            if any(k in entry for k in keys):
                print({k: round(entry[k], 4) for k in keys if k in entry})
    except Exception as e:
        print(f"重型 cell 未能运行（缺依赖或无网络均属正常）：{type(e).__name__}: {e}")

## ✏️ 练习 1：实现数值稳定的 `dpo_loss`

从公式 $\mathcal{L} = -\log\sigma\big(\beta[(\log\pi_\theta(y_w)-\log\pi_{\text{ref}}(y_w)) - (\log\pi_\theta(y_l)-\log\pi_{\text{ref}}(y_l))]\big)$ 出发，
对 batch 取均值。

**提示**：
- 必须用 `F.logsigmoid(x)`，**不要**写 `torch.log(torch.sigmoid(x))`——后者在 margin 很负时 `sigmoid≈0`，`log(0) = -inf`；
- 三行以内可以写完；
- 想想 $\beta$ 和 margin 的乘法关系意味着什么缩放性质（自测会考）。

In [ ]:
def dpo_loss(logp_c, logp_r, ref_logp_c, ref_logp_r, beta):
    # 输入均为形状 [B] 的张量；返回标量 loss
    # TODO: 1) 计算 margin = beta * ((logp_c - ref_logp_c) - (logp_r - ref_logp_r))
    # TODO: 2) 返回 (-F.logsigmoid(margin)) 的均值
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
z = torch.zeros(3)
# margin = 0 时 loss = -log(1/2) = log 2
assert torch.allclose(dpo_loss(z, z, z, z, beta=0.1), torch.tensor(0.69314718), atol=1e-5)
# margin -> +inf 时 loss -> 0
big = torch.tensor([100.0])
assert dpo_loss(big, -big, torch.zeros(1), torch.zeros(1), beta=0.1).item() < 1e-6
# 数值稳定性：极端负 margin 不得出现 inf/nan（log(sigmoid(...)) 的写法在这里会挂）
bad = dpo_loss(torch.tensor([-2000.0]), torch.tensor([2000.0]),
               torch.zeros(1), torch.zeros(1), beta=1.0)
assert torch.isfinite(bad), "极端 margin 下数值不稳定——你是不是用了 log(sigmoid(x)) ?"
# 缩放性质：loss 只依赖 beta * h，所以 (beta, h) 与 (beta/2, 2h) 等价
d = torch.tensor([1.7, -0.3, 0.4])
l1 = dpo_loss(d,     torch.zeros(3), torch.zeros(3), torch.zeros(3), beta=0.2)
l2 = dpo_loss(2 * d, torch.zeros(3), torch.zeros(3), torch.zeros(3), beta=0.1)
assert torch.allclose(l1, l2, atol=1e-6)
print("✅ 练习 1 通过")

## ✏️ 练习 2：隐式奖励 `implicit_reward` 与 `reward_margin`

DPO 的隐式奖励 $\hat r_\theta(x,y) = \beta\log\frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$（差一个只依赖 $x$ 的常数）。
trl 日志里的 `rewards/chosen`、`rewards/rejected`、`rewards/margins` 就是这几个量。实现：

- `implicit_reward(logp, ref_logp, beta)` → $\beta(\log\pi_\theta - \log\pi_{\text{ref}})$；
- `reward_margin(logp_c, logp_r, ref_logp_c, ref_logp_r, beta)` → $\hat r(y_w) - \hat r(y_l)$（请**复用** `implicit_reward`）。

**提示**：自测会验证 $\mathcal{L}_{\mathrm{DPO}} = -\log\sigma(\text{reward\_margin})$ 的一致性——margin 正是 DPO loss 内部那个量。

In [ ]:
def implicit_reward(logp, ref_logp, beta):
    # TODO: 返回 beta * (logp - ref_logp)
    raise NotImplementedError

def reward_margin(logp_c, logp_r, ref_logp_c, ref_logp_r, beta):
    # TODO: 用 implicit_reward 计算 chosen 与 rejected 的隐式奖励之差
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测 ----
# 与参考 policy 完全一致 => 隐式奖励为 0
assert torch.allclose(implicit_reward(torch.tensor([-1.0]), torch.tensor([-1.0]), 0.3),
                      torch.zeros(1))
lc  = torch.tensor([-1.0, -2.0]); lr2 = torch.tensor([-3.0, -1.5])
rc  = torch.tensor([-1.5, -1.5]); rr  = torch.tensor([-2.5, -2.0])
m = reward_margin(lc, lr2, rc, rr, beta=0.1)
# 手算第一条：0.1 * ((-1.0+1.5) - (-3.0+2.5)) = 0.1 * (0.5 + 0.5) = 0.1
assert torch.allclose(m[0], torch.tensor(0.1), atol=1e-6)
# 一致性：-logsigmoid(reward_margin) 的均值 == DPO loss（loss 内部就是这个量）
expected = -F.logsigmoid(0.1 * ((lc - rc) - (lr2 - rr))).mean()
assert torch.allclose(-F.logsigmoid(m).mean(), expected, atol=1e-6)
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `ipo_loss`

IPO [Azar 2023] 把 logistic 换成平方损失：
$$\mathcal{L}_{\mathrm{IPO}} = \Big(h - \frac{1}{2\beta}\Big)^2,\qquad
h = (\log\pi_\theta(y_w)-\log\pi_{\text{ref}}(y_w)) - (\log\pi_\theta(y_l)-\log\pi_{\text{ref}}(y_l))$$

对 batch 取均值。与 DPO 的本质区别：最优 margin 是**有限值** $\tfrac{1}{2\beta}$，超过会被往回拉。

**提示**：两行可写完；自测会做数值扫描，验证 loss 在 $h = \tfrac{1}{2\beta}$ 处取最小。

In [ ]:
def ipo_loss(logp_c, logp_r, ref_logp_c, ref_logp_r, beta):
    # TODO: 1) 计算 h（注意：不乘 beta！）
    # TODO: 2) 返回 ((h - 1/(2*beta)) ** 2) 的均值
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测：数值扫描验证最优 margin ----
beta = 0.1
target = 1.0 / (2.0 * beta)        # = 5.0
zeros1 = torch.zeros(1)
margins = torch.linspace(0.0, 10.0, 101)
losses = torch.stack([ipo_loss(torch.tensor([m_]), zeros1, zeros1, zeros1, beta)
                      for m_ in margins])
best = margins[losses.argmin()].item()
assert abs(best - target) < 0.11, f"最优 margin 应在 1/(2*beta)={target}, 扫描得到 {best}"
# 恰在目标处 loss = 0
assert ipo_loss(torch.tensor([target]), zeros1, zeros1, zeros1, beta).item() < 1e-8
# 两侧都会被惩罚（DPO 做不到这一点：margin 偏大时 DPO loss 反而更小）
assert ipo_loss(torch.tensor([target - 2.0]), zeros1, zeros1, zeros1, beta) > 1.0
assert ipo_loss(torch.tensor([target + 2.0]), zeros1, zeros1, zeros1, beta) > 1.0
print("✅ 练习 3 通过")

## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def dpo_loss(logp_c, logp_r, ref_logp_c, ref_logp_r, beta):
    margin = beta * ((logp_c - ref_logp_c) - (logp_r - ref_logp_r))
    return -F.logsigmoid(margin).mean()

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def implicit_reward(logp, ref_logp, beta):
    return beta * (logp - ref_logp)

def reward_margin(logp_c, logp_r, ref_logp_c, ref_logp_r, beta):
    return implicit_reward(logp_c, ref_logp_c, beta) - implicit_reward(logp_r, ref_logp_r, beta)

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def ipo_loss(logp_c, logp_r, ref_logp_c, ref_logp_r, beta):
    h = (logp_c - ref_logp_c) - (logp_r - ref_logp_r)
    return ((h - 1.0 / (2.0 * beta)) ** 2).mean()

## 小结

| 实验 | 现象 | 对应讲解 |
|---|---|---|
| DPO/IPO/SimPO 对比 | DPO margin 无界增长；IPO 被回归目标拉住；SimPO 无 ref 也能对齐 | 第 2、4 节 |
| chosen 概率下降 | $\log\pi(y_w)$ 与 $\log\pi(y_l)$ 双双下跌、差距拉大——loss 只看相对量 | 第 3 节 |
| β 扫描 | 解析最优的 KL 随 $\beta$ 单调递减；固定预算下小 $\beta$ 漂移更远，但梯度尺度 $\propto\beta$ 使「目标」与「速度」纠缠 | 第 5 节 |
| trl 实跑（可选） | `rewards/margins` 上升、两个 rewards 同时变负是正常形态 | 第 5 节 |

**记住验收铁律**：margin/accuracy 全是判别式代理指标，上线前必须做独立的生成式评测 + 长度对照（讲解第 7 节）。

**下一步 → 模块 05 · 推理模型与 RLVR/GRPO**：当任务有客观对错（数学/代码），偏好数据可以被
**可验证奖励**取代——在线 RL 重新登场，但这次不需要奖励模型，也不需要 value 网络。

---
## 🎯 真实数据胶囊题：真实偏好上的 DPO 损失

DPO 不训练奖励模型，直接用偏好对优化策略。用真实红酒质量对（高质量=chosen），实现 DPO loss，验证：当 chosen 相对 rejected 的 logprob 优势增大时，loss 下降。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.post_training_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=200):
    p=_fetch("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    rows=[json.loads(l) for l in open(p).read().splitlines()[:n]]
    return rows
def winequality():
    import pandas as pd
    p=_fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv","winequality-red.csv")
    return pd.read_csv(p, sep=";")

df = winequality()
rng=np.random.default_rng(0)
# 真实偏好对：随机取两瓶，质量高的为 chosen
idx=rng.integers(0,len(df),(500,2)); q=df["quality"].to_numpy()
pairs=[(a,b) if q[a]>=q[b] else (b,a) for a,b in idx]
print(f"{len(pairs)} 个真实偏好对 (chosen 质量 >= rejected)")

**练习**：实现 `dpo_loss(lp_chosen, lp_rejected, ref_chosen, ref_rejected, beta)`，DPO 目标 `-log sigmoid(beta·((lp_c-ref_c)-(lp_r-ref_r)))`（平均）。

In [ ]:
def dpo_loss(lp_chosen, lp_rejected, ref_chosen, ref_rejected, beta=0.1):
    # TODO: margin = beta*((lp_c-ref_c)-(lp_r-ref_r)); loss = -mean(log sigmoid(margin))
    raise NotImplementedError


In [ ]:
# 自测
n=len(pairs); rng=np.random.default_rng(1)
ref_c=rng.normal(size=n); ref_r=rng.normal(size=n)
# 情形A: 策略让 chosen 更高（margin 大）
lp_c_good=ref_c+1.0; lp_r_good=ref_r-1.0
# 情形B: 策略让 rejected 更高（坏）
lp_c_bad=ref_c-1.0; lp_r_bad=ref_r+1.0
loss_good=dpo_loss(lp_c_good,lp_r_good,ref_c,ref_r)
loss_bad =dpo_loss(lp_c_bad, lp_r_bad, ref_c,ref_r)
assert loss_good < loss_bad, "更偏好 chosen 时 DPO loss 应更低"
assert loss_good > 0
print(f"DPO ✓  偏好chosen时 loss={loss_good:.3f} < 偏好rejected时 {loss_bad:.3f}")


### 📖 参考答案

In [ ]:
def dpo_loss(lp_chosen, lp_rejected, ref_chosen, ref_rejected, beta=0.1):
    margin = beta*((lp_chosen-ref_chosen)-(lp_rejected-ref_rejected))
    return float(-np.log(1/(1+np.exp(-margin))).mean())
print("✓ DPO 把 RLHF 变成一个简单的分类损失，不需要单独的 reward model")